In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Reproducibility Note

The previous experiment notebook was used for exploration and model testing. It included several CNN architectures, image sizes, learning rates, and training setups. However, that notebook was not fully seeded, so the results were not completely reproducible across reruns.

CNN training includes several sources of randomness, including weight initialization, training data shuffling, dropout, image augmentation, and some GPU operations. Because of this, rerunning the same model can produce slightly different validation and test results.

To make this final notebook more reliable, this notebook sets a fixed random seed before splitting the data, creating DataLoaders, training the model, and evaluating results. The final reported results in this notebook should come from this seeded run rather than from earlier unseeded experiments.

In [ ]:
# Reproducibility setup

import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Seed set to:", SEED)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
nutrition_path = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/dish_nutrition_values.csv"
IMAGE_ROOT = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset/imagery/realsense_overhead"

In [ ]:
print("Nutrition CSV exists:", os.path.exists(nutrition_path))
print("Image root exists:", os.path.exists(IMAGE_ROOT))

In [ ]:
nutrition_df = pd.read_csv(nutrition_path)

def get_rgb_path(dish_id):
    return os.path.join(IMAGE_ROOT, dish_id, "rgb.png")

nutrition_df["image_path"] = nutrition_df["dish_id"].apply(get_rgb_path)
nutrition_df["image_exists"] = nutrition_df["image_path"].apply(os.path.exists)

print("Original dataframe shape:", nutrition_df.shape)
print(nutrition_df["image_exists"].value_counts())

nutrition_df.head()

In [ ]:
clean_model_df = nutrition_df.dropna(
    subset=["calories", "mass", "fat", "carb", "protein"]
).copy()

clean_model_df = clean_model_df[clean_model_df["image_exists"] == True].copy()

clean_model_df["calorie_quantile_class"] = pd.qcut(
    clean_model_df["calories"],
    q=3,
    labels=[0, 1, 2]
).astype(int)

class_names_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

clean_model_df["calorie_quantile_label"] = clean_model_df["calorie_quantile_class"].map(class_names_map)

print("Clean model dataframe shape:", clean_model_df.shape)
print(clean_model_df["calorie_quantile_label"].value_counts())

clean_model_df.head()

In [ ]:
multitask_df = clean_model_df.dropna(
    subset=[
        "image_path",
        "calorie_quantile_class",
        "calories",
        "mass",
        "fat",
        "carb",
        "protein"
    ]
).copy()

train_df_mt, temp_df_mt = train_test_split(
    multitask_df,
    test_size=0.30,
    random_state=SEED,
    stratify=multitask_df["calorie_quantile_class"]
)

val_df_mt, test_df_mt = train_test_split(
    temp_df_mt,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df_mt["calorie_quantile_class"]
)

print("Train:", train_df_mt.shape)
print("Validation:", val_df_mt.shape)
print("Test:", test_df_mt.shape)

print("\nTrain class balance:")
print(train_df_mt["calorie_quantile_label"].value_counts())

print("\nValidation class balance:")
print(val_df_mt["calorie_quantile_label"].value_counts())

print("\nTest class balance:")
print(test_df_mt["calorie_quantile_label"].value_counts())

In [ ]:
regression_targets = ["calories", "mass", "fat", "carb", "protein"]

scaler = StandardScaler()

train_df_mt_scaled = train_df_mt.copy()
val_df_mt_scaled = val_df_mt.copy()
test_df_mt_scaled = test_df_mt.copy()

train_df_mt_scaled[regression_targets] = scaler.fit_transform(
    train_df_mt[regression_targets]
)

val_df_mt_scaled[regression_targets] = scaler.transform(
    val_df_mt[regression_targets]
)

test_df_mt_scaled[regression_targets] = scaler.transform(
    test_df_mt[regression_targets]
)

train_df_mt_scaled[regression_targets].head()

In [ ]:
train_transform_mt = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),
    transforms.ToTensor()
])

eval_transform_mt = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
class NutritionMultiTaskDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.loc[idx, "image_path"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        class_label = int(self.df.loc[idx, "calorie_quantile_class"])

        regression_values = self.df.loc[
            idx,
            ["calories", "mass", "fat", "carb", "protein"]
        ].values.astype("float32")

        regression_values = torch.tensor(regression_values, dtype=torch.float32)

        return image, class_label, regression_values

In [ ]:
train_dataset_mt = NutritionMultiTaskDataset(
    train_df_mt_scaled,
    transform=train_transform_mt
)

val_dataset_mt = NutritionMultiTaskDataset(
    val_df_mt_scaled,
    transform=eval_transform_mt
)

test_dataset_mt = NutritionMultiTaskDataset(
    test_df_mt_scaled,
    transform=eval_transform_mt
)

g = torch.Generator()
g.manual_seed(SEED)

train_loader_mt = DataLoader(
    train_dataset_mt,
    batch_size=32,
    shuffle=True,
    generator=g
)

val_loader_mt = DataLoader(
    val_dataset_mt,
    batch_size=32,
    shuffle=False
)

test_loader_mt = DataLoader(
    test_dataset_mt,
    batch_size=32,
    shuffle=False
)

In [ ]:
images, class_labels, regression_values = next(iter(train_loader_mt))

print("Images:", images.shape)
print("Class labels:", class_labels.shape)
print("Regression values:", regression_values.shape)

In [ ]:
class NutritionOrdinalMultiTaskDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.loc[idx, "image_path"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        class_label = int(self.df.loc[idx, "calorie_quantile_class"])

        # Ordinal encoding:
        # Low    = 0 -> [0, 0]
        # Medium = 1 -> [1, 0]
        # High   = 2 -> [1, 1]
        ordinal_label = torch.tensor(
            [
                1.0 if class_label >= 1 else 0.0,
                1.0 if class_label >= 2 else 0.0
            ],
            dtype=torch.float32
        )

        regression_values = self.df.loc[
            idx,
            ["calories", "mass", "fat", "carb", "protein"]
        ].values.astype("float32")

        regression_values = torch.tensor(regression_values, dtype=torch.float32)

        return image, ordinal_label, regression_values, class_label

In [ ]:
train_dataset_ordinal = NutritionOrdinalMultiTaskDataset(
    train_df_mt_scaled,
    transform=train_transform_mt
)

val_dataset_ordinal = NutritionOrdinalMultiTaskDataset(
    val_df_mt_scaled,
    transform=eval_transform_mt
)

test_dataset_ordinal = NutritionOrdinalMultiTaskDataset(
    test_df_mt_scaled,
    transform=eval_transform_mt
)

g = torch.Generator()
g.manual_seed(SEED)

train_loader_ordinal = DataLoader(
    train_dataset_ordinal,
    batch_size=32,
    shuffle=True,
    generator=g
)

val_loader_ordinal = DataLoader(
    val_dataset_ordinal,
    batch_size=32,
    shuffle=False
)

test_loader_ordinal = DataLoader(
    test_dataset_ordinal,
    batch_size=32,
    shuffle=False
)

In [ ]:
images, ordinal_labels, regression_values, class_labels = next(iter(train_loader_ordinal))

print("Images:", images.shape)
print("Ordinal labels:", ordinal_labels.shape)
print("Regression values:", regression_values.shape)
print("Class labels:", class_labels.shape)

print("\nExample class labels:")
print(class_labels[:10])

print("\nExample ordinal labels:")
print(ordinal_labels[:10])

## Model Setup

The next section defines the CNN architecture. The baseline multi-task model uses a standard 3-class classification head. The ordinal experiment replaces that with a 2-output ordinal head:

- Low: [0, 0]
- Medium: [1, 0]
- High: [1, 1]

This lets the model learn calorie thresholds instead of treating Low, Medium, and High as unrelated categories.

## Ordinal Multi-Task CNN

This model uses the same image feature extractor as the extra-layer CNN, but replaces the standard 3-class classification head with a 2-output ordinal head.

Instead of treating Low, Medium, and High as unrelated classes, the ordinal model learns two thresholds:

- Output 1: Is the meal above Low?
- Output 2: Is the meal above Medium?

The ordinal labels are encoded as:

- Low: [0, 0]
- Medium: [1, 0]
- High: [1, 1]

The model still uses the auxiliary regression task to predict calories, mass, fat, carbohydrates, and protein during training.

In [ ]:
class ExtraLayerOrdinalMultiTaskCNN224(nn.Module):
    def __init__(self):
        super(ExtraLayerOrdinalMultiTaskCNN224, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # 224 -> 112 -> 56 -> 28 -> 14 after four pooling layers
        self.shared_fc = nn.Linear(128 * 14 * 14, 128)
        self.dropout = nn.Dropout(0.3)

        # Two ordinal threshold outputs
        self.ordinal_head = nn.Linear(128, 2)

        # Auxiliary regression output:
        # calories, mass, fat, carb, protein
        self.regression_head = nn.Linear(128, 5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 224 -> 112
        x = self.pool(F.relu(self.conv2(x)))  # 112 -> 56
        x = self.pool(F.relu(self.conv3(x)))  # 56 -> 28
        x = self.pool(F.relu(self.conv4(x)))  # 28 -> 14

        x = x.view(x.size(0), -1)

        features = F.relu(self.shared_fc(x))
        features = self.dropout(features)

        ordinal_output = self.ordinal_head(features)
        regression_output = self.regression_head(features)

        return ordinal_output, regression_output

In [ ]:
def ordinal_logits_to_class(ordinal_logits):
    probabilities = torch.sigmoid(ordinal_logits)

    # Each passed threshold adds 1 to the predicted class.
    # [0, 0] -> 0 = Low
    # [1, 0] -> 1 = Medium
    # [1, 1] -> 2 = High
    passed_thresholds = (probabilities >= 0.5).int()

    predicted_classes = passed_thresholds.sum(dim=1)

    return predicted_classes

In [ ]:
def train_ordinal_multitask_model(weight=0.5, lr=0.0005, epochs=25):
    print("\nTraining ordinal multi-task CNN")
    print(f"regression_loss_weight = {weight}")
    print(f"learning_rate = {lr}\n")

    model = ExtraLayerOrdinalMultiTaskCNN224().to(device)

    ordinal_loss_fn = nn.BCEWithLogitsLoss()
    regression_loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_accuracy = 0.0
    best_model_path = "/kaggle/working/best_ordinal_multitask_224.pth"

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, ordinal_labels, regression_values, class_labels in train_loader_ordinal:
            images = images.to(device)
            ordinal_labels = ordinal_labels.to(device)
            regression_values = regression_values.to(device)
            class_labels = class_labels.to(device)

            optimizer.zero_grad()

            ordinal_outputs, regression_outputs = model(images)

            ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)
            reg_loss = regression_loss_fn(regression_outputs, regression_values)

            loss = ordinal_loss + weight * reg_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            predicted_classes = ordinal_logits_to_class(ordinal_outputs)

            total += class_labels.size(0)
            correct += (predicted_classes == class_labels).sum().item()

        train_loss = running_loss / len(train_loader_ordinal)
        train_accuracy = correct / total

        model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, ordinal_labels, regression_values, class_labels in val_loader_ordinal:
                images = images.to(device)
                ordinal_labels = ordinal_labels.to(device)
                regression_values = regression_values.to(device)
                class_labels = class_labels.to(device)

                ordinal_outputs, regression_outputs = model(images)

                ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)
                reg_loss = regression_loss_fn(regression_outputs, regression_values)

                loss = ordinal_loss + weight * reg_loss

                val_running_loss += loss.item()

                predicted_classes = ordinal_logits_to_class(ordinal_outputs)

                val_total += class_labels.size(0)
                val_correct += (predicted_classes == class_labels).sum().item()

        val_loss = val_running_loss / len(val_loader_ordinal)
        val_accuracy = val_correct / val_total

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), best_model_path)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_accuracy:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_accuracy:.4f}, "
            f"Best Val Acc: {best_val_accuracy:.4f}"
        )

    return model, best_model_path, best_val_accuracy, train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
ordinal_model, ordinal_model_path, ordinal_best_val, ordinal_train_losses, ordinal_val_losses, ordinal_train_acc, ordinal_val_acc = train_ordinal_multitask_model(
    weight=0.5,
    lr=0.0005,
    epochs=25
)

In [ ]:
ordinal_model.load_state_dict(torch.load(ordinal_model_path))
ordinal_model.eval()

all_preds_ordinal = []
all_labels_ordinal = []

with torch.no_grad():
    for images, ordinal_labels, regression_values, class_labels in test_loader_ordinal:
        images = images.to(device)
        class_labels = class_labels.to(device)

        ordinal_outputs, regression_outputs = ordinal_model(images)

        predicted_classes = ordinal_logits_to_class(ordinal_outputs)

        all_preds_ordinal.extend(predicted_classes.cpu().numpy())
        all_labels_ordinal.extend(class_labels.cpu().numpy())

test_accuracy_ordinal = sum(
    pred == label for pred, label in zip(all_preds_ordinal, all_labels_ordinal)
) / len(all_labels_ordinal)

print("Ordinal Multi-Task CNN Test Accuracy:", test_accuracy_ordinal)
print("Ordinal Multi-Task CNN Best Validation Accuracy:", ordinal_best_val)

In [ ]:
class_names = ["Low", "Medium", "High"]

print(classification_report(
    all_labels_ordinal,
    all_preds_ordinal,
    target_names=class_names
))

In [ ]:
cm_ordinal = confusion_matrix(
    all_labels_ordinal,
    all_preds_ordinal
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_ordinal,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Ordinal Multi-Task CNN Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(ordinal_train_losses, label="Train Loss")
plt.plot(ordinal_val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Ordinal Multi-Task CNN Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(ordinal_train_acc, label="Train Accuracy")
plt.plot(ordinal_val_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Ordinal Multi-Task CNN Accuracy")
plt.legend()
plt.show()

In [ ]:
baseline_accuracy = 0.7228 

print("Baseline extra-layer multi-task CNN accuracy:", baseline_accuracy)
print("Ordinal multi-task CNN accuracy:", test_accuracy_ordinal)
print("Ordinal best validation accuracy:", ordinal_best_val)

if test_accuracy_ordinal > baseline_accuracy:
    print("The ordinal model improved over the baseline.")
else:
    print("The ordinal model did not improve over the baseline.")

## Ordinal Model Results

The ordinal multi-task CNN achieved the best performance in the seeded notebook run. It reached **74.1% test accuracy**, compared to **72.3%** for the baseline extra-layer multi-task CNN.

This result suggests that representing calorie classes as ordered categories improved model performance. Instead of treating Low, Medium, and High as unrelated classes, the ordinal model learned two thresholds: whether a meal is above Low and whether a meal is above Medium.

The model achieved a best validation accuracy of **77.6%** and a test accuracy of **74.1%**. The final F1-scores were **0.81** for Low, **0.61** for Medium, and **0.79** for High. Medium remained the hardest class, but the ordinal model improved overall accuracy compared with the seeded baseline.


In [ ]:
class ExtraLayerOrdinalSeparateHeadsCNN224(nn.Module):
    def __init__(self):
        super(ExtraLayerOrdinalSeparateHeadsCNN224, self).__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        # 224 -> 112 -> 56 -> 28 -> 14 after four pooling layers
        self.shared_fc = nn.Linear(128 * 14 * 14, 128)
        self.dropout = nn.Dropout(0.3)

        # Main task: ordinal calorie classification
        self.ordinal_head = nn.Linear(128, 2)

        # Auxiliary tasks: separate regression heads
        self.calorie_head = nn.Linear(128, 1)
        self.mass_head = nn.Linear(128, 1)
        self.fat_head = nn.Linear(128, 1)
        self.carb_head = nn.Linear(128, 1)
        self.protein_head = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 224 -> 112
        x = self.pool(F.relu(self.conv2(x)))  # 112 -> 56
        x = self.pool(F.relu(self.conv3(x)))  # 56 -> 28
        x = self.pool(F.relu(self.conv4(x)))  # 28 -> 14

        x = x.view(x.size(0), -1)

        features = F.relu(self.shared_fc(x))
        features = self.dropout(features)

        ordinal_output = self.ordinal_head(features)

        calorie_output = self.calorie_head(features)
        mass_output = self.mass_head(features)
        fat_output = self.fat_head(features)
        carb_output = self.carb_head(features)
        protein_output = self.protein_head(features)

        regression_outputs = {
            "calories": calorie_output,
            "mass": mass_output,
            "fat": fat_output,
            "carb": carb_output,
            "protein": protein_output
        }

        return ordinal_output, regression_outputs

In [ ]:
def train_ordinal_separate_heads_model(weight=0.5, lr=0.0005, epochs=25):
    print("\nTraining ordinal multi-task CNN with separate regression heads")
    print(f"regression_loss_weight = {weight}")
    print(f"learning_rate = {lr}\n")

    model = ExtraLayerOrdinalSeparateHeadsCNN224().to(device)

    ordinal_loss_fn = nn.BCEWithLogitsLoss()
    regression_loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_accuracy = 0.0
    best_model_path = "/kaggle/working/best_ordinal_separate_heads_224.pth"

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, ordinal_labels, regression_values, class_labels in train_loader_ordinal:
            images = images.to(device)
            ordinal_labels = ordinal_labels.to(device)
            regression_values = regression_values.to(device)
            class_labels = class_labels.to(device)

            optimizer.zero_grad()

            ordinal_outputs, regression_outputs = model(images)

            ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)

            calorie_loss = regression_loss_fn(regression_outputs["calories"], regression_values[:, 0].unsqueeze(1))
            mass_loss = regression_loss_fn(regression_outputs["mass"], regression_values[:, 1].unsqueeze(1))
            fat_loss = regression_loss_fn(regression_outputs["fat"], regression_values[:, 2].unsqueeze(1))
            carb_loss = regression_loss_fn(regression_outputs["carb"], regression_values[:, 3].unsqueeze(1))
            protein_loss = regression_loss_fn(regression_outputs["protein"], regression_values[:, 4].unsqueeze(1))

            reg_loss = (
                calorie_loss +
                mass_loss +
                fat_loss +
                carb_loss +
                protein_loss
            ) / 5

            loss = ordinal_loss + weight * reg_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            predicted_classes = ordinal_logits_to_class(ordinal_outputs)

            total += class_labels.size(0)
            correct += (predicted_classes == class_labels).sum().item()

        train_loss = running_loss / len(train_loader_ordinal)
        train_accuracy = correct / total

        model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, ordinal_labels, regression_values, class_labels in val_loader_ordinal:
                images = images.to(device)
                ordinal_labels = ordinal_labels.to(device)
                regression_values = regression_values.to(device)
                class_labels = class_labels.to(device)

                ordinal_outputs, regression_outputs = model(images)

                ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)

                calorie_loss = regression_loss_fn(regression_outputs["calories"], regression_values[:, 0].unsqueeze(1))
                mass_loss = regression_loss_fn(regression_outputs["mass"], regression_values[:, 1].unsqueeze(1))
                fat_loss = regression_loss_fn(regression_outputs["fat"], regression_values[:, 2].unsqueeze(1))
                carb_loss = regression_loss_fn(regression_outputs["carb"], regression_values[:, 3].unsqueeze(1))
                protein_loss = regression_loss_fn(regression_outputs["protein"], regression_values[:, 4].unsqueeze(1))

                reg_loss = (
                    calorie_loss +
                    mass_loss +
                    fat_loss +
                    carb_loss +
                    protein_loss
                ) / 5

                loss = ordinal_loss + weight * reg_loss

                val_running_loss += loss.item()

                predicted_classes = ordinal_logits_to_class(ordinal_outputs)

                val_total += class_labels.size(0)
                val_correct += (predicted_classes == class_labels).sum().item()

        val_loss = val_running_loss / len(val_loader_ordinal)
        val_accuracy = val_correct / val_total

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), best_model_path)

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_accuracy:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_accuracy:.4f}, "
            f"Best Val Acc: {best_val_accuracy:.4f}"
        )

    return model, best_model_path, best_val_accuracy, train_losses, val_losses, train_accuracies, val_accuracies

In [ ]:
separate_heads_model, separate_heads_model_path, separate_heads_best_val, separate_heads_train_losses, separate_heads_val_losses, separate_heads_train_acc, separate_heads_val_acc = train_ordinal_separate_heads_model(
    weight=0.5,
    lr=0.0005,
    epochs=25
)

In [ ]:
separate_heads_model.load_state_dict(torch.load(separate_heads_model_path))
separate_heads_model.eval()

all_preds_separate_heads = []
all_labels_separate_heads = []

with torch.no_grad():
    for images, ordinal_labels, regression_values, class_labels in test_loader_ordinal:
        images = images.to(device)
        class_labels = class_labels.to(device)

        ordinal_outputs, regression_outputs = separate_heads_model(images)

        predicted_classes = ordinal_logits_to_class(ordinal_outputs)

        all_preds_separate_heads.extend(predicted_classes.cpu().numpy())
        all_labels_separate_heads.extend(class_labels.cpu().numpy())

test_accuracy_separate_heads = sum(
    pred == label for pred, label in zip(all_preds_separate_heads, all_labels_separate_heads)
) / len(all_labels_separate_heads)

print("Ordinal Separate-Heads Multi-Task CNN Test Accuracy:", test_accuracy_separate_heads)
print("Ordinal Separate-Heads Best Validation Accuracy:", separate_heads_best_val)

In [ ]:
class_names = ["Low", "Medium", "High"]

print(classification_report(
    all_labels_separate_heads,
    all_preds_separate_heads,
    target_names=class_names
))

In [ ]:
cm_separate_heads = confusion_matrix(
    all_labels_separate_heads,
    all_preds_separate_heads
)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_separate_heads,
    annot=True,
    fmt="d",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Ordinal Separate-Heads Multi-Task CNN Confusion Matrix")
plt.show()

In [ ]:
ordinal_accuracy = 0.7413

print("Ordinal combined regression head accuracy:", ordinal_accuracy)
print("Ordinal separate regression heads accuracy:", test_accuracy_separate_heads)
print("Ordinal separate regression heads best validation accuracy:", separate_heads_best_val)

difference = test_accuracy_separate_heads - ordinal_accuracy

print("Difference:", difference)

if test_accuracy_separate_heads > ordinal_accuracy:
    print("The separate-head model improved over the combined-head ordinal model.")
elif test_accuracy_separate_heads == ordinal_accuracy:
    print("The separate-head model matched the combined-head ordinal model.")
else:
    print("The separate-head model did not improve over the combined-head ordinal model.")

In [ ]:
def continue_training_separate_heads_model(
    model,
    current_best_val,
    weight=0.5,
    lr=0.00025,
    epochs=5
):
    print("\nContinuing separate-head ordinal model training")
    print(f"regression_loss_weight = {weight}")
    print(f"learning_rate = {lr}\n")

    model = model.to(device)

    ordinal_loss_fn = nn.BCEWithLogitsLoss()
    regression_loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_accuracy = current_best_val
    best_model_path = "/kaggle/working/best_ordinal_separate_heads_224_extended.pth"

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, ordinal_labels, regression_values, class_labels in train_loader_ordinal:
            images = images.to(device)
            ordinal_labels = ordinal_labels.to(device)
            regression_values = regression_values.to(device)
            class_labels = class_labels.to(device)

            optimizer.zero_grad()

            ordinal_outputs, regression_outputs = model(images)

            ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)

            calorie_loss = regression_loss_fn(regression_outputs["calories"], regression_values[:, 0].unsqueeze(1))
            mass_loss = regression_loss_fn(regression_outputs["mass"], regression_values[:, 1].unsqueeze(1))
            fat_loss = regression_loss_fn(regression_outputs["fat"], regression_values[:, 2].unsqueeze(1))
            carb_loss = regression_loss_fn(regression_outputs["carb"], regression_values[:, 3].unsqueeze(1))
            protein_loss = regression_loss_fn(regression_outputs["protein"], regression_values[:, 4].unsqueeze(1))

            reg_loss = (
                calorie_loss +
                mass_loss +
                fat_loss +
                carb_loss +
                protein_loss
            ) / 5

            loss = ordinal_loss + weight * reg_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            predicted_classes = ordinal_logits_to_class(ordinal_outputs)

            total += class_labels.size(0)
            correct += (predicted_classes == class_labels).sum().item()

        train_loss = running_loss / len(train_loader_ordinal)
        train_accuracy = correct / total

        model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, ordinal_labels, regression_values, class_labels in val_loader_ordinal:
                images = images.to(device)
                ordinal_labels = ordinal_labels.to(device)
                regression_values = regression_values.to(device)
                class_labels = class_labels.to(device)

                ordinal_outputs, regression_outputs = model(images)

                ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)

                calorie_loss = regression_loss_fn(regression_outputs["calories"], regression_values[:, 0].unsqueeze(1))
                mass_loss = regression_loss_fn(regression_outputs["mass"], regression_values[:, 1].unsqueeze(1))
                fat_loss = regression_loss_fn(regression_outputs["fat"], regression_values[:, 2].unsqueeze(1))
                carb_loss = regression_loss_fn(regression_outputs["carb"], regression_values[:, 3].unsqueeze(1))
                protein_loss = regression_loss_fn(regression_outputs["protein"], regression_values[:, 4].unsqueeze(1))

                reg_loss = (
                    calorie_loss +
                    mass_loss +
                    fat_loss +
                    carb_loss +
                    protein_loss
                ) / 5

                loss = ordinal_loss + weight * reg_loss

                val_running_loss += loss.item()

                predicted_classes = ordinal_logits_to_class(ordinal_outputs)

                val_total += class_labels.size(0)
                val_correct += (predicted_classes == class_labels).sum().item()

        val_loss = val_running_loss / len(val_loader_ordinal)
        val_accuracy = val_correct / val_total

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), best_model_path)

        print(
            f"Extra Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_accuracy:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_accuracy:.4f}, "
            f"Best Val Acc: {best_val_accuracy:.4f}"
        )

    return model, best_model_path, best_val_accuracy

In [ ]:
separate_heads_model, separate_heads_extended_path, separate_heads_extended_best_val = continue_training_separate_heads_model(
    separate_heads_model,
    current_best_val=separate_heads_best_val,
    weight=0.5,
    lr=0.00025,
    epochs=5
)

In [ ]:
separate_heads_model.load_state_dict(torch.load(separate_heads_extended_path))
separate_heads_model.eval()

all_preds_separate_heads_extended = []
all_labels_separate_heads_extended = []

with torch.no_grad():
    for images, ordinal_labels, regression_values, class_labels in test_loader_ordinal:
        images = images.to(device)
        class_labels = class_labels.to(device)

        ordinal_outputs, regression_outputs = separate_heads_model(images)

        predicted_classes = ordinal_logits_to_class(ordinal_outputs)

        all_preds_separate_heads_extended.extend(predicted_classes.cpu().numpy())
        all_labels_separate_heads_extended.extend(class_labels.cpu().numpy())

test_accuracy_separate_heads_extended = sum(
    pred == label
    for pred, label in zip(all_preds_separate_heads_extended, all_labels_separate_heads_extended)
) / len(all_labels_separate_heads_extended)

print("Extended Separate-Heads Test Accuracy:", test_accuracy_separate_heads_extended)
print("Extended Separate-Heads Best Val Accuracy:", separate_heads_extended_best_val)

print(classification_report(
    all_labels_separate_heads_extended,
    all_preds_separate_heads_extended,
    target_names=["Low", "Medium", "High"]
))

In [ ]:
separate_heads_model.load_state_dict(torch.load(separate_heads_extended_path))
separate_heads_model.eval()

all_preds_separate_heads_extended = []
all_labels_separate_heads_extended = []

with torch.no_grad():
    for images, ordinal_labels, regression_values, class_labels in test_loader_ordinal:
        images = images.to(device)
        class_labels = class_labels.to(device)

        ordinal_outputs, regression_outputs = separate_heads_model(images)

        predicted_classes = ordinal_logits_to_class(ordinal_outputs)

        all_preds_separate_heads_extended.extend(predicted_classes.cpu().numpy())
        all_labels_separate_heads_extended.extend(class_labels.cpu().numpy())

test_accuracy_separate_heads_extended = sum(
    pred == label
    for pred, label in zip(all_preds_separate_heads_extended, all_labels_separate_heads_extended)
) / len(all_labels_separate_heads_extended)

print("Extended Separate-Heads Test Accuracy:", test_accuracy_separate_heads_extended)
print("Extended Separate-Heads Best Val Accuracy:", separate_heads_extended_best_val)

print(classification_report(
    all_labels_separate_heads_extended,
    all_preds_separate_heads_extended,
    target_names=["Low", "Medium", "High"]
))

In [ ]:
combined_head_ordinal_accuracy = 0.7412731006160165

print("Combined-head ordinal accuracy:", combined_head_ordinal_accuracy)
print("Extended separate-head ordinal accuracy:", test_accuracy_separate_heads_extended)

difference = test_accuracy_separate_heads_extended - combined_head_ordinal_accuracy

print("Difference:", difference)

if test_accuracy_separate_heads_extended > combined_head_ordinal_accuracy:
    print("The extended separate-head model is the new best model.")
else:
    print("The combined-head ordinal model remains the best model.")

## Extended Separate Regression Heads Results

The separate-head ordinal model was trained for 25 epochs and then fine-tuned for 5 additional epochs with a smaller learning rate. The additional training improved the best validation accuracy from 75.8% to 77.4%.

However, the extended separate-head model achieved 72.5% test accuracy, which was lower than both the original separate-head model and the combined-head ordinal model. This suggests that the extra training improved validation performance but did not improve generalization to unseen test data.

Because the combined-head ordinal model achieved the highest test accuracy at 74.1%, it remained the final selected model.

In [ ]:
def fine_tune_ordinal_model(
    model,
    current_best_val,
    weight=0.5,
    lr=0.00025,
    epochs=5
):
    print("\nFine-tuning ordinal multi-task CNN")
    print(f"regression_loss_weight = {weight}")
    print(f"learning_rate = {lr}\n")

    model = model.to(device)

    ordinal_loss_fn = nn.BCEWithLogitsLoss()
    regression_loss_fn = nn.MSELoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_accuracy = current_best_val
    best_model_path = "/kaggle/working/best_ordinal_multitask_224_finetuned.pth"

    for epoch in range(epochs):
        model.train()

        running_loss = 0.0
        correct = 0
        total = 0

        for images, ordinal_labels, regression_values, class_labels in train_loader_ordinal:
            images = images.to(device)
            ordinal_labels = ordinal_labels.to(device)
            regression_values = regression_values.to(device)
            class_labels = class_labels.to(device)

            optimizer.zero_grad()

            ordinal_outputs, regression_outputs = model(images)

            ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)
            reg_loss = regression_loss_fn(regression_outputs, regression_values)

            loss = ordinal_loss + weight * reg_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            predicted_classes = ordinal_logits_to_class(ordinal_outputs)

            total += class_labels.size(0)
            correct += (predicted_classes == class_labels).sum().item()

        train_loss = running_loss / len(train_loader_ordinal)
        train_accuracy = correct / total

        model.eval()

        val_running_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, ordinal_labels, regression_values, class_labels in val_loader_ordinal:
                images = images.to(device)
                ordinal_labels = ordinal_labels.to(device)
                regression_values = regression_values.to(device)
                class_labels = class_labels.to(device)

                ordinal_outputs, regression_outputs = model(images)

                ordinal_loss = ordinal_loss_fn(ordinal_outputs, ordinal_labels)
                reg_loss = regression_loss_fn(regression_outputs, regression_values)

                loss = ordinal_loss + weight * reg_loss

                val_running_loss += loss.item()

                predicted_classes = ordinal_logits_to_class(ordinal_outputs)

                val_total += class_labels.size(0)
                val_correct += (predicted_classes == class_labels).sum().item()

        val_loss = val_running_loss / len(val_loader_ordinal)
        val_accuracy = val_correct / val_total

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), best_model_path)

        print(
            f"Fine-tune Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_accuracy:.4f}, "
            f"Val Loss: {val_loss:.4f}, "
            f"Val Acc: {val_accuracy:.4f}, "
            f"Best Val Acc: {best_val_accuracy:.4f}"
        )

    return model, best_model_path, best_val_accuracy

In [ ]:
ordinal_model = ExtraLayerOrdinalMultiTaskCNN224().to(device)
ordinal_model.load_state_dict(torch.load("/kaggle/working/best_ordinal_multitask_224.pth"))
ordinal_model.eval()

In [ ]:
ordinal_model, ordinal_finetuned_path, ordinal_finetuned_best_val = fine_tune_ordinal_model(
    ordinal_model,
    current_best_val=ordinal_best_val,
    weight=0.5,
    lr=0.00025,
    epochs=5
)

In [ ]:
ordinal_model.load_state_dict(torch.load(ordinal_finetuned_path))
ordinal_model.eval()

all_preds_ordinal_finetuned = []
all_labels_ordinal_finetuned = []

with torch.no_grad():
    for images, ordinal_labels, regression_values, class_labels in test_loader_ordinal:
        images = images.to(device)
        class_labels = class_labels.to(device)

        ordinal_outputs, regression_outputs = ordinal_model(images)

        predicted_classes = ordinal_logits_to_class(ordinal_outputs)

        all_preds_ordinal_finetuned.extend(predicted_classes.cpu().numpy())
        all_labels_ordinal_finetuned.extend(class_labels.cpu().numpy())

test_accuracy_ordinal_finetuned = sum(
    pred == label
    for pred, label in zip(all_preds_ordinal_finetuned, all_labels_ordinal_finetuned)
) / len(all_labels_ordinal_finetuned)

print("Original Ordinal Test Accuracy: 0.7412731006160165")
print("Fine-Tuned Ordinal Test Accuracy:", test_accuracy_ordinal_finetuned)
print("Fine-Tuned Best Validation Accuracy:", ordinal_finetuned_best_val)

print(classification_report(
    all_labels_ordinal_finetuned,
    all_preds_ordinal_finetuned,
    target_names=["Low", "Medium", "High"]
))

## Fine-Tuning Results

After the ordinal multi-task CNN achieved 74.1% test accuracy, it was fine-tuned for 5 additional epochs using a smaller learning rate of 0.00025. The best validation accuracy improved from 77.6% to 78.2%, showing that the model continued to improve on the validation set.

However, the fine-tuned model achieved 73.7% test accuracy, which was slightly lower than the original ordinal model's 74.1% test accuracy. This suggests that the additional fine-tuning did not improve generalization to unseen test data.

Because the original ordinal model achieved the highest test accuracy, it remained the final selected model.

In [ ]:
import os

final_model_path = "/kaggle/working/best_ordinal_multitask_224.pth"

print("Final checkpoint exists:", os.path.exists(final_model_path))
print("Final checkpoint path:", final_model_path)

In [ ]:
import torch
import joblib

final_package_path = "/kaggle/working/final_ordinal_multitask_model.pth"
scaler_path = "/kaggle/working/regression_target_scaler.pkl"

# Load the best original ordinal checkpoint
final_model = ExtraLayerOrdinalMultiTaskCNN224().to(device)
final_model.load_state_dict(torch.load("/kaggle/working/best_ordinal_multitask_224.pth"))
final_model.eval()

# Save full model package
torch.save({
    "model_state_dict": final_model.state_dict(),
    "model_name": "ExtraLayerOrdinalMultiTaskCNN224",
    "image_size": 224,
    "class_names": ["Low", "Medium", "High"],
    "ordinal_encoding": {
        "Low": [0, 0],
        "Medium": [1, 0],
        "High": [1, 1]
    },
    "regression_targets": ["calories", "mass", "fat", "carb", "protein"],
    "test_accuracy": 0.7412731006160165,
    "best_validation_accuracy": 0.7761806981519507,
    "learning_rate": 0.0005,
    "regression_loss_weight": 0.5,
    "epochs": 25,
    "seed": SEED
}, final_package_path)

# Save scaler too, useful if you want to interpret regression outputs later
joblib.dump(scaler, scaler_path)

print("Saved final model package to:", final_package_path)
print("Saved scaler to:", scaler_path)

In [ ]:
checkpoint = torch.load("/kaggle/working/final_ordinal_multitask_model.pth", map_location=device)

loaded_model = ExtraLayerOrdinalMultiTaskCNN224().to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

print("Loaded model:", checkpoint["model_name"])
print("Test accuracy:", checkpoint["test_accuracy"])
print("Class names:", checkpoint["class_names"])

In [ ]:
import joblib

scaler_path = "/kaggle/working/regression_target_scaler.pkl"

joblib.dump(scaler, scaler_path)

print("Saved scaler to:", scaler_path)

In [ ]:
import os

print("Model exists:", os.path.exists("/kaggle/working/final_ordinal_multitask_model.pth"))
print("Scaler exists:", os.path.exists("/kaggle/working/regression_target_scaler.pkl"))

In [ ]:
checkpoint = torch.load("/kaggle/working/final_ordinal_multitask_model.pth", map_location=device)

loaded_model = ExtraLayerOrdinalMultiTaskCNN224().to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

print("Loaded model:", checkpoint["model_name"])
print("Test accuracy:", checkpoint["test_accuracy"])
print("Class names:", checkpoint["class_names"])